#### Mouse gene programs
- Reusable biology-known validation framework
- This script validates gene programs against public annotation databases.

- Take each program gene list.
- Query enrichment resources such as GO Biological Process, GO Cellular Component, Reactome, KEGG, WikiPathways, Mouse Gene Atlas, MGI-related annotations, etc.
- Ask: “Do the enriched terms match the biological label I gave this program?”
- Produce a report showing: enriched terms, adjusted p-values, overlapping genes, whether the enrichment supports, weakly supports, or does not support the claimed interpretation.

- This is not perfect, because enrichment databases are biased toward well-studied genes. But it is the correct scriptable way to sanity-check whether the biology of the program is defensible.


In [1]:
import os
import numpy as np
import pandas as pd
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
commonpathlocal = "/home/jfpoyatos/Work/MNCN_24_25/A - plasticity - What's next/"
commonpath = "scripts/data/Mouse/"

**Helper functions**

In [3]:
def get_available_libraries(candidate_libraries, organism="Mouse"):
    """
    Return only Enrichr libraries that are currently available.
    This avoids repeated runtime warnings from unavailable libraries.
    """
    try:
        available = gp.get_library_name(organism=organism)
    except Exception as e:
        print(f"[WARNING] Could not fetch available Enrichr libraries: {e}")
        print("[WARNING] Proceeding with candidate libraries as provided.")
        return candidate_libraries

    available_set = set(available)
    valid = [lib for lib in candidate_libraries if lib in available_set]
    missing = [lib for lib in candidate_libraries if lib not in available_set]

    if missing:
        print("\nUnavailable libraries will be skipped:")
        for lib in missing:
            print(f"  - {lib}")

    print("\nLibraries used:")
    for lib in valid:
        print(f"  - {lib}")

    return valid


In [4]:
def clean_term(term):
    if pd.isna(term):
        return ""
    return str(term).lower()


def keyword_hit(term, keywords):
    term_low = clean_term(term)
    hits = [kw for kw in keywords if kw.lower() in term_low]
    return hits

In [5]:
def validate_terms_for_program(df, program_name, expected_keywords):
    """
    Add keyword-matching columns to an enrichment result table.
    """
    if df.empty:
        return df

    keywords = expected_keywords.get(program_name, [])

    df = df.copy()
    df["expected_keywords"] = ", ".join(keywords)
    df["matched_keywords"] = df["Term"].apply(
        lambda x: ", ".join(keyword_hit(x, keywords))
    )
    df["has_expected_keyword"] = df["matched_keywords"].str.len() > 0

    return df

In [6]:
def run_enrichment_for_program(program_name, genes, libraries, organism="Mouse"):
    """
    Run Enrichr enrichment for one gene program across multiple libraries.
    """
    all_results = []

    for lib in libraries:
        try:
            enr = gp.enrichr(
                gene_list=genes,
                gene_sets=lib,
                organism=organism,
                outdir=None,
                cutoff=1.0,
                verbose=False
            )

            if enr is None or enr.results is None or enr.results.empty:
                continue

            res = enr.results.copy()
            res["program"] = program_name
            res["library"] = lib
            res["input_genes"] = ", ".join(genes)
            res["n_input_genes"] = len(genes)

            all_results.append(res)

        except Exception as e:
            print(f"[WARNING] Enrichment failed for {program_name} / {lib}: {e}")

    if len(all_results) == 0:
        return pd.DataFrame()

    return pd.concat(all_results, ignore_index=True)

In [7]:
def summarize_program_support(df, program_name, pval_cutoff=0.05):
    """
    Summarize whether a program has significant terms matching the expected biology.
    """
    if df.empty:
        return {
            "program": program_name,
            "n_terms_total": 0,
            "n_significant_terms": 0,
            "n_significant_expected_terms": 0,
            "best_expected_term": None,
            "best_expected_library": None,
            "best_expected_adj_p": np.nan,
            "best_expected_overlap": None,
            "best_expected_genes": None,
            "support_level": "no enrichment result",
            "interpretation": (
                "No enrichment result was returned. Check gene symbols, "
                "program size, and library availability."
            )
        }

    sig = df[df["Adjusted P-value"] < pval_cutoff].copy()
    sig_expected = sig[sig["has_expected_keyword"]].copy()

    if sig_expected.shape[0] > 0:
        sig_expected = sig_expected.sort_values("Adjusted P-value")
        best = sig_expected.iloc[0]

        return {
            "program": program_name,
            "n_terms_total": df.shape[0],
            "n_significant_terms": sig.shape[0],
            "n_significant_expected_terms": sig_expected.shape[0],
            "best_expected_term": best["Term"],
            "best_expected_library": best["library"],
            "best_expected_adj_p": best["Adjusted P-value"],
            "best_expected_overlap": best.get("Overlap", None),
            "best_expected_genes": best.get("Genes", None),
            "support_level": "strong support",
            "interpretation": (
                "At least one significantly enriched term matches the expected biology."
            )
        }

    if sig.shape[0] > 0:
        sig = sig.sort_values("Adjusted P-value")
        best = sig.iloc[0]

        return {
            "program": program_name,
            "n_terms_total": df.shape[0],
            "n_significant_terms": sig.shape[0],
            "n_significant_expected_terms": 0,
            "best_expected_term": None,
            "best_expected_library": None,
            "best_expected_adj_p": np.nan,
            "best_expected_overlap": None,
            "best_expected_genes": None,
            "top_significant_term": best["Term"],
            "top_significant_library": best["library"],
            "top_significant_adj_p": best["Adjusted P-value"],
            "support_level": "ambiguous support",
            "interpretation": (
                "Significant enrichments were detected, but they did not clearly "
                "match the expected biological keywords."
            )
        }

    return {
        "program": program_name,
        "n_terms_total": df.shape[0],
        "n_significant_terms": 0,
        "n_significant_expected_terms": 0,
        "best_expected_term": None,
        "best_expected_library": None,
        "best_expected_adj_p": np.nan,
        "best_expected_overlap": None,
        "best_expected_genes": None,
        "support_level": "weak/no support",
        "interpretation": (
            "No significant terms were detected at adjusted P < "
            f"{pval_cutoff}. This may happen for small or weakly annotated gene sets."
        )
    }


In [8]:
def compute_program_overlap(programs):
    """
    Compute pairwise gene-set overlap and Jaccard index.
    """
    names = list(programs.keys())
    rows = []

    for a in names:
        for b in names:
            set_a = set(programs[a])
            set_b = set(programs[b])
            inter = set_a.intersection(set_b)
            union = set_a.union(set_b)

            rows.append({
                "program_a": a,
                "program_b": b,
                "n_a": len(set_a),
                "n_b": len(set_b),
                "n_overlap": len(inter),
                "jaccard": len(inter) / len(union) if len(union) > 0 else np.nan,
                "overlapping_genes": ", ".join(sorted(inter))
            })

    return pd.DataFrame(rows)

In [9]:
def save_program_input(programs, outdir):
    """
    Save input gene programs as CSV.
    """
    rows = []

    for program, genes in programs.items():
        rows.append({
            "program": program,
            "n_genes": len(genes),
            "genes": ", ".join(genes)
        })

    df = pd.DataFrame(rows)

    df.to_csv(
        os.path.join(outdir, "01_gene_programs_input.csv"),
        index=False
    )

    return df

In [10]:
def save_overlap_outputs(overlap_df, outdir, title="Program gene-set overlap"):
    """
    Save overlap table and heatmap.
    """
    overlap_df.to_csv(
        os.path.join(outdir, "program_gene_overlap.csv"),
        index=False
    )

    overlap_pivot = overlap_df.pivot(
        index="program_a",
        columns="program_b",
        values="jaccard"
    )

    n_programs = overlap_pivot.shape[0]
    fig_size = max(8, min(24, 0.35 * n_programs))

    plt.figure(figsize=(fig_size, fig_size))
    sns.heatmap(overlap_pivot, cmap="viridis", square=True)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, "program_gene_overlap_heatmap.pdf"),
        dpi=300
    )
    plt.close()

In [11]:
def save_validation_heatmap(summary_df, outdir, title="Biology-known validation support"):
    """
    Save support-level heatmap.
    """
    support_rank = {
        "strong support": 3,
        "ambiguous support": 2,
        "weak/no support": 1,
        "no enrichment result": 0,
    }

    summary_df = summary_df.copy()
    summary_df["support_score"] = summary_df["support_level"].map(support_rank)

    plot_df = summary_df[["program", "support_score"]].set_index("program")

    plt.figure(figsize=(6, max(8, 0.28 * plot_df.shape[0])))
    sns.heatmap(
        plot_df,
        annot=True,
        cmap="YlGnBu",
        cbar_kws={"label": "Support score"},
        vmin=0,
        vmax=3
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, "05_validation_heatmap.pdf"),
        dpi=300
    )
    plt.close()

    return summary_df


In [12]:
def add_warning(warnings_rows, program, issue, recommendation):
    warnings_rows.append({
        "program": program,
        "issue": issue,
        "recommendation": recommendation
    })


def get_jaccard(overlap_df, a, b):
    vals = overlap_df[
        (overlap_df["program_a"] == a) &
        (overlap_df["program_b"] == b)
    ]["jaccard"].values

    if len(vals) == 0:
        return np.nan

    return vals[0]

In [13]:
def make_warnings_from_rules(overlap_df, programs, warning_rules):
    """
    Generate wording warnings from overlap-based and composition-based rules.

    warning_rules format:
        [
            {
                "type": "overlap",
                "program_a": "...",
                "program_b": "...",
                "threshold": 0.25,
                "recommendation": "..."
            },
            {
                "type": "contains_any",
                "program": "...",
                "genes": ["..."],
                "recommendation": "..."
            }
        ]
    """
    warnings_rows = []

    for rule in warning_rules:
        rule_type = rule.get("type")

        if rule_type == "overlap":
            a = rule["program_a"]
            b = rule["program_b"]
            threshold = rule.get("threshold", 0.25)

            jac = get_jaccard(overlap_df, a, b)

            if not np.isnan(jac) and jac >= threshold:
                add_warning(
                    warnings_rows,
                    f"{a} / {b}",
                    f"Gene-set overlap Jaccard = {jac:.2f}",
                    rule["recommendation"]
                )

        elif rule_type == "contains_any":
            program = rule["program"]
            genes = set(rule["genes"])
            program_genes = set(programs.get(program, []))
            hits = sorted(program_genes.intersection(genes))

            if len(hits) > 0:
                add_warning(
                    warnings_rows,
                    program,
                    f"Contains genes requiring cautious interpretation: {', '.join(hits)}",
                    rule["recommendation"]
                )

        elif rule_type == "contains_all":
            program = rule["program"]
            genes = set(rule["genes"])
            program_genes = set(programs.get(program, []))

            if genes.issubset(program_genes):
                add_warning(
                    warnings_rows,
                    program,
                    f"Contains full cautionary gene set: {', '.join(sorted(genes))}",
                    rule["recommendation"]
                )

        else:
            print(f"[WARNING] Unknown warning rule type: {rule_type}")

    return pd.DataFrame(warnings_rows)

In [14]:
def run_program_validation(
    programs,
    expected_keywords,
    candidate_libraries,
    warning_rules,
    outdir,
    organism="Mouse",
    pval_cutoff=0.05,
    title_prefix="Program"
):
    """
    Main reusable pipeline.
    """
    os.makedirs(outdir, exist_ok=True)

    save_program_input(programs, outdir)

    overlap_df = compute_program_overlap(programs)
    save_overlap_outputs(
        overlap_df,
        outdir,
        title=f"{title_prefix} gene-set overlap: Jaccard index"
    )

    libraries = get_available_libraries(
        candidate_libraries,
        organism=organism
    )

    if len(libraries) == 0:
        raise RuntimeError(
            "No requested Enrichr libraries were available. "
            "Check gseapy installation or Enrichr connectivity."
        )

    all_enrichment_results = []

    for program_name, genes in programs.items():
        print(f"Running enrichment for: {program_name}")

        res = run_enrichment_for_program(
            program_name=program_name,
            genes=genes,
            libraries=libraries,
            organism=organism
        )

        if not res.empty:
            res = validate_terms_for_program(
                res,
                program_name=program_name,
                expected_keywords=expected_keywords
            )

            all_enrichment_results.append(res)

    if len(all_enrichment_results) > 0:
        all_enrichment_results = pd.concat(
            all_enrichment_results,
            ignore_index=True
        )
    else:
        all_enrichment_results = pd.DataFrame()

    if not all_enrichment_results.empty:
        all_enrichment_results.to_csv(
            os.path.join(outdir, "02_all_enrichment_results.csv"),
            index=False
        )

    summary_rows = []

    for program_name in programs.keys():
        if all_enrichment_results.empty:
            df_prog = pd.DataFrame()
        else:
            df_prog = all_enrichment_results[
                all_enrichment_results["program"] == program_name
            ].copy()

        summary_rows.append(
            summarize_program_support(
                df_prog,
                program_name,
                pval_cutoff=pval_cutoff
            )
        )

    summary_df = pd.DataFrame(summary_rows)

    summary_df.to_csv(
        os.path.join(outdir, "03_program_validation_summary.csv"),
        index=False
    )

    if not all_enrichment_results.empty:
        top_terms = (
            all_enrichment_results
            .sort_values(["program", "Adjusted P-value"])
            .groupby("program")
            .head(10)
            .reset_index(drop=True)
        )

        top_terms.to_csv(
            os.path.join(outdir, "04_top_terms_by_program.csv"),
            index=False
        )

    summary_df = save_validation_heatmap(
        summary_df,
        outdir,
        title=f"{title_prefix} biology-known validation support"
    )

    warnings_df = make_warnings_from_rules(
        overlap_df=overlap_df,
        programs=programs,
        warning_rules=warning_rules
    )

    warnings_df.to_csv(
        os.path.join(outdir, "06_wording_warnings.csv"),
        index=False
    )

    print("\nProgram validation summary:")
    print(summary_df[[
        "program",
        "support_level",
        "n_significant_terms",
        "n_significant_expected_terms",
        "best_expected_term",
        "best_expected_adj_p"
    ]])

    print("\nWording warnings:")
    print(warnings_df)

    print("\nDone.")
    print(f"Results written to: {outdir}")
    print("\nOpen these files first:")
    print("  03_program_validation_summary.csv")
    print("  04_top_terms_by_program.csv")
    print("  06_wording_warnings.csv")
    print("  program_gene_overlap.csv")
    print("  05_validation_heatmap.pdf")

    return {
        "summary": summary_df,
        "overlap": overlap_df,
        "warnings": warnings_df,
        "enrichment": all_enrichment_results,
        "libraries_used": libraries,
    }

#### Cancer programs

In [ ]:
# these are the programs reported

PROGRAMS_MOUSE_CANCER = {
    "emt_invasion": [
        "Vim", "Fn1", "Zeb1", "Zeb2", "Snai1", "Snai2", "Twist1",
        "Cdh2", "Itga5", "Itgav", "Itgb1", "Mmp2", "Mmp9", "Mmp14",
        "Col1a1", "Col1a2", "Col3a1", "Sparc", "Tnc", "Lox", "Postn"
    ],

    "epithelial_identity": [
        "Epcam", "Cdh1", "Krt8", "Krt18", "Krt19", "Krt7",
        "Cldn3", "Cldn4", "Cldn7", "Ocln", "Tjp1", "Muc1"
    ],

    "inflammatory_signaling": [
        "Nfkb1", "Nfkb2", "Rela", "Relb", "Tnf", "Tnfaip3",
        "Il1a", "Il1b", "Il6", "Cxcl1", "Cxcl2", "Cxcl10",
        "Ccl2", "Ccl5", "Ptgs2", "Socs3", "Icam1"
    ],

    "interferon_response": [
        "Stat1", "Stat2", "Irf1", "Irf7", "Irf9", "Isg15",
        "Ifit1", "Ifit2", "Ifit3", "Ifitm1", "Ifitm2", "Ifitm3",
        "Oas1a", "Oas1g", "Oas2", "Oas3", "Mx1", "Rsad2",
        "Usp18", "Dhx58"
    ],

    "hypoxia": [
        "Hif1a", "Vegfa", "Slc2a1", "Ldha", "Pdk1", "Eno1",
        "Aldoa", "Bnip3", "Ndrg1", "Adm", "Car9", "Serpine1",
        "Ero1a", "Ankrd37"
    ],

    "oxidative_stress": [
        "Nfe2l2", "Keap1", "Hmox1", "Nqo1", "Gclc", "Gclm",
        "Gsr", "Gpx2", "Gpx4", "Prdx1", "Txn1", "Txnrd1",
        "Srxn1", "Sqstm1", "Ftl1", "Fth1"
    ],

    "unfolded_protein_response": [
        "Hspa5", "Hsp90b1", "Ddit3", "Atf3", "Atf4", "Atf6",
        "Xbp1", "Ern1", "Eif2ak3", "Pdia3", "Pdia4", "Pdia6",
        "Dnajb9", "Herpud1", "Derl3", "Manf", "Creld2"
    ],

    "KRAS_mapk_signaling": [
        "Dusp4", "Dusp5", "Dusp6", "Etv1", "Etv4", "Etv5",
        "Spry1", "Spry2", "Spry4", "Fos", "Fosl1", "Jun",
        "Junb", "Egr1", "Egr2", "Ets1", "Myc", "Mapk1", "Mapk3"
    ],

    "myc_targets": [
        "Myc", "Max", "Mxd1", "Npm1", "Ncl", "Odc1", "Cad",
        "Nme1", "Nme2", "Hspd1", "Hspe1", "Tfrc", "Slc2a1",
        "Phb", "Pa2g4", "Hnrnpa1", "Hnrnpa2b1", "Eif4a1"
    ],

    "p53_pathway": [
        "Trp53", "Cdkn1a", "Mdm2", "Bax", "Bbc3", "Pmaip1",
        "Gadd45a", "Gadd45g", "Sesn1", "Sesn2", "Ddb2",
        "Phlda3", "Zmat3", "Rrm2b", "Ccng1"
    ],

    "cell_cycle": [
        "Mki67", "Top2a", "Pcna", "Mcm2", "Mcm3", "Mcm4",
        "Mcm5", "Mcm6", "Mcm7", "Cdk1", "Ccnb1", "Ccnb2",
        "Cenpa", "Cenpe", "Cenpf", "Tpx2", "Bub1b", "Ube2c",
        "Aspm", "Nusap1", "Kif11", "Kif20b"
    ],

    "dna_damage_repair": [
        "Brca1", "Brca2", "Rad51", "Rad51ap1", "Msh2", "Msh6",
        "Pms2", "Pcna", "Rpa1", "Rpa2", "Chek1", "Chek2",
        "Atm", "Atr", "Fanca", "Fancc", "Xrcc5", "Xrcc6"
    ],

    "glycolysis_metabolic_rewiring": [
        "Slc2a1", "Hk1", "Hk2", "Pfkl", "Pfkp", "Aldoa",
        "Gapdh", "Pgam1", "Eno1", "Pkm", "Ldha", "Pdk1", "Slc16a3"
    ],

    "oxidative_phospho_mitochondrial": [
        "Ndufa2", "Ndufa4", "Ndufb3", "Ndufb5", "Sdha", "Sdhb",
        "Uqcrc1", "Uqcrc2", "Cyc1", "Cox4i1", "Cox5a",
        "Atp5f1a", "Atp5f1b", "Atp5mc1"
    ],

    "ecm_remodeling_invasion": [
        "Fn1", "Col1a1", "Col1a2", "Col3a1", "Col4a1", "Col4a2",
        "Col5a1", "Col5a2", "Sparc", "Tnc", "Postn", "Lox",
        "Loxl2", "Mmp2", "Mmp9", "Mmp14", "Itga5", "Itgav",
        "Itgb1", "Vcan", "Thbs1"
    ],

    "antigen_presentation_mhc_i": [
        "B2m", "H2-K1", "H2-D1", "Tap1", "Tap2", "Tapbp",
        "Psmb8", "Psmb9", "Psmb10", "Nlrc5"
    ],

    "antigen_presentation_mhc_ii": [
        "H2-Aa", "H2-Ab1", "H2-Eb1", "Cd74", "Ciita",
        "H2-DMa", "H2-DMb1", "H2-Ob"
    ],

    "stemness_progenitor_like": [
        "Sox9", "Prom1", "Aldh1a1", "Aldh1a3", "Lgr5",
        "Krt19", "Epcam", "Tacstd2", "Muc1", "Nes",
        "Vim", "Lgals1", "S100a10"
    ],

    "therapy_resist_stress_tolerance": [
        "Aldh1a1", "Aldh1a3", "Abcb1a", "Abcb1b", "Abcc1",
        "Abcc3", "Abcg2", "Bcl2", "Bcl2l1", "Mcl1",
        "Hspa1a", "Hspa1b", "Hspa5", "Hmox1", "Nqo1",
        "Sod2", "Gpx2", "Lgals1", "Tgfb1", "Vim"
    ],

    "tgfb_signaling": [
        "Tgfb1", "Tgfb2", "Tgfbr1", "Tgfbr2", "Smad2",
        "Smad3", "Smad4", "Serpine1", "Id1", "Id2",
        "Id3", "Skil", "Snai1", "Snai2"
    ],

    "apoptosis": [
        "Bax", "Bak1", "Bbc3", "Pmaip1", "Casp3", "Casp7",
        "Casp8", "Casp9", "Apaf1", "Bid", "Bad",
        "Bcl2", "Bcl2l1", "Mcl1"
    ],

    "senescence_sasp": [
        "Cdkn1a", "Cdkn2a", "Trp53", "Il6", "Cxcl1",
        "Cxcl2", "Ccl2", "Serpine1", "Mmp3", "Mmp9",
        "Igfbp3", "Igfbp5", "Gadd45a"
    ],

    "ap1_immediate_early_response": [
        "Fos", "Fosb", "Fosl1", "Fosl2", "Jun", "Junb",
        "Jund", "Atf3", "Egr1", "Egr2", "Dusp1",
        "Dusp5", "Dusp6", "Ier2", "Csrnp1", "Zfp36",
        "Btg2", "Ccn1", "Klf6"
    ],
}

In [ ]:
# Extended programs = diagnostic controls that decompose the originals.
# # These are useful for checking whether a broad program is driven
# # by the specific biological component or by a shared stress/EMT/
# # proliferation backbone.

PROGRAMS_MOUSE_CANCER_EXTENDED = {
    **PROGRAMS_MOUSE_CANCER,

    "emt_tf_core": [
        "Zeb1", "Zeb2", "Snai1", "Snai2", "Twist1"
    ],

    "emt_mesenchymal_core": [
        "Vim", "Fn1", "Cdh2", "Itga5", "Itgav", "Itgb1"
    ],

    "ecm_collagen_core": [
        "Col1a1", "Col1a2", "Col3a1", "Col4a1",
        "Col4a2", "Col5a1", "Col5a2"
    ],

    "ecm_protease_invasion_core": [
        "Mmp2", "Mmp9", "Mmp14", "Lox", "Loxl2"
    ],

    "nfkb_cytokine_core": [
        "Nfkb1", "Nfkb2", "Rela", "Relb", "Tnf",
        "Il1a", "Il1b", "Il6", "Tnfaip3"
    ],

    "chemokine_core": [
        "Cxcl1", "Cxcl2", "Cxcl10", "Ccl2", "Ccl5"
    ],

    "interferon_tf_core": [
        "Stat1", "Stat2", "Irf1", "Irf7", "Irf9"
    ],

    "interferon_isg_core": [
        "Isg15", "Ifit1", "Ifit2", "Ifit3", "Ifitm1",
        "Ifitm2", "Ifitm3", "Oas1a", "Oas1g", "Oas2",
        "Oas3", "Mx1", "Rsad2", "Usp18", "Dhx58"
    ],

    "hif_hypoxia_core": [
        "Hif1a", "Vegfa", "Bnip3", "Ndrg1",
        "Adm", "Car9", "Serpine1", "Ankrd37"
    ],

    "hypoxia_glycolysis_component": [
        "Slc2a1", "Ldha", "Pdk1", "Eno1", "Aldoa"
    ],

    "nrf2_antioxidant_core": [
        "Nfe2l2", "Keap1", "Hmox1", "Nqo1",
        "Gclc", "Gclm", "Gsr", "Srxn1", "Sqstm1"
    ],

    "glutathione_redox_core": [
        "Gclc", "Gclm", "Gsr", "Gpx2", "Gpx4",
        "Prdx1", "Txn1", "Txnrd1"
    ],

    "upr_sensor_tf_core": [
        "Ddit3", "Atf3", "Atf4", "Atf6",
        "Xbp1", "Ern1", "Eif2ak3"
    ],

    "er_chaperone_folding_core": [
        "Hspa5", "Hsp90b1", "Pdia3", "Pdia4",
        "Pdia6", "Dnajb9", "Herpud1", "Derl3",
        "Manf", "Creld2"
    ],

    "ras_mapk_feedback_core": [
        "Dusp4", "Dusp5", "Dusp6", "Spry1",
        "Spry2", "Spry4"
    ],

    "ap1_core": [
        "Fos", "Fosb", "Fosl1", "Fosl2",
        "Jun", "Junb", "Jund", "Atf3"
    ],

    "myc_biosynthesis_core": [
        "Myc", "Max", "Npm1", "Ncl", "Odc1",
        "Cad", "Nme1", "Nme2", "Tfrc", "Eif4a1"
    ],

    "p53_arrest_apoptosis_core": [
        "Trp53", "Cdkn1a", "Mdm2", "Bax", "Bbc3",
        "Pmaip1", "Gadd45a", "Gadd45g", "Ddb2"
    ],

    "s_phase_replication_core": [
        "Pcna", "Mcm2", "Mcm3", "Mcm4", "Mcm5",
        "Mcm6", "Mcm7"
    ],

    "g2m_mitotic_core": [
        "Mki67", "Top2a", "Cdk1", "Ccnb1", "Ccnb2",
        "Cenpa", "Cenpe", "Cenpf", "Tpx2", "Bub1b",
        "Ube2c", "Aspm", "Nusap1", "Kif11", "Kif20b"
    ],

    "homologous_recombination_core": [
        "Brca1", "Brca2", "Rad51", "Rad51ap1", "Rpa1", "Rpa2"
    ],

    "mismatch_checkpoint_repair_core": [
        "Msh2", "Msh6", "Pms2", "Chek1", "Chek2", "Atm", "Atr"
    ],

    "glycolysis_core": [
        "Slc2a1", "Hk1", "Hk2", "Pfkl", "Pfkp",
        "Aldoa", "Gapdh", "Pgam1", "Eno1", "Pkm", "Ldha"
    ],

    "lactate_pyruvate_core": [
        "Ldha", "Pdk1", "Slc16a3"
    ],

    "mitochondrial_complex_i_core": [
        "Ndufa2", "Ndufa4", "Ndufb3", "Ndufb5"
    ],

    "mitochondrial_etc_atp_core": [
        "Sdha", "Sdhb", "Uqcrc1", "Uqcrc2", "Cyc1",
        "Cox4i1", "Cox5a", "Atp5f1a", "Atp5f1b", "Atp5mc1"
    ],

    "mhc_i_core": [
        "B2m", "H2-K1", "H2-D1", "Tap1", "Tap2",
        "Tapbp", "Psmb8", "Psmb9", "Psmb10", "Nlrc5"
    ],

    "mhc_ii_core": [
        "H2-Aa", "H2-Ab1", "H2-Eb1", "Cd74",
        "Ciita", "H2-DMa", "H2-DMb1", "H2-Ob"
    ],

    "stemness_epithelial_core": [
        "Sox9", "Prom1", "Aldh1a1", "Aldh1a3",
        "Lgr5", "Krt19", "Epcam", "Tacstd2", "Muc1"
    ],

    "stemness_plasticity_component": [
        "Nes", "Vim", "Lgals1", "S100a10"
    ],

    "drug_efflux_core": [
        "Abcb1a", "Abcb1b", "Abcc1", "Abcc3", "Abcg2"
    ],

    "anti_apoptotic_resistance_core": [
        "Bcl2", "Bcl2l1", "Mcl1"
    ],

    "stress_tolerance_core": [
        "Hspa1a", "Hspa1b", "Hspa5", "Hmox1",
        "Nqo1", "Sod2", "Gpx2"
    ],

    "tgfb_smad_core": [
        "Tgfb1", "Tgfb2", "Tgfbr1", "Tgfbr2",
        "Smad2", "Smad3", "Smad4"
    ],

    "tgfb_emt_response_core": [
        "Serpine1", "Id1", "Id2", "Id3",
        "Skil", "Snai1", "Snai2"
    ],

    "intrinsic_apoptosis_core": [
        "Bax", "Bak1", "Bbc3", "Pmaip1",
        "Casp9", "Apaf1", "Bid", "Bad"
    ],

    "executioner_caspase_core": [
        "Casp3", "Casp7", "Casp8", "Casp9"
    ],

    "anti_apoptotic_buffering_core": [
        "Bcl2", "Bcl2l1", "Mcl1"
    ],

    "senescence_arrest_core": [
        "Cdkn1a", "Cdkn2a", "Trp53", "Gadd45a"
    ],

    "sasp_inflammatory_core": [
        "Il6", "Cxcl1", "Cxcl2", "Ccl2",
        "Mmp3", "Mmp9", "Igfbp3", "Igfbp5", "Serpine1"
    ],

    "immediate_early_tf_core": [
        "Fos", "Fosb", "Fosl1", "Fosl2",
        "Jun", "Junb", "Jund", "Atf3",
        "Egr1", "Egr2", "Klf6"
    ],

    "mapk_feedback_immediate_core": [
        "Dusp1", "Dusp5", "Dusp6", "Ier2",
        "Csrnp1", "Zfp36", "Btg2", "Ccn1"
    ],
}


In [ ]:
# Expected biological keywords
EXPECTED_KEYWORDS_CANCER = {
    "emt_invasion": [
        "epithelial mesenchymal", "emt", "mesenchymal", "migration",
        "motility", "invasion", "extracellular matrix", "collagen",
        "matrix", "adhesion", "wound"
    ],

    "epithelial_identity": [
        "epithelial", "epithelium", "cell junction", "tight junction",
        "adherens", "keratin", "cell adhesion", "polarity"
    ],

    "inflammatory_signaling": [
        "inflammatory", "inflammation", "nf-kappa", "nfkb",
        "cytokine", "chemokine", "tnf", "interleukin", "immune response"
    ],

    "interferon_response": [
        "interferon", "antiviral", "response to virus", "stat",
        "irf", "isg", "type i interferon", "type ii interferon"
    ],

    "hypoxia": [
        "hypoxia", "oxygen", "hif", "angiogenesis", "vegf",
        "anaerobic", "response to decreased oxygen"
    ],

    "oxidative_stress": [
        "oxidative stress", "reactive oxygen", "antioxidant",
        "glutathione", "detoxification", "nrf2", "redox",
        "heme", "iron"
    ],

    "unfolded_protein_response": [
        "unfolded protein", "endoplasmic reticulum", "er stress",
        "protein folding", "chaperone", "atf", "xbp1", "perk"
    ],

    "KRAS_mapk_signaling": [
        "ras", "mapk", "erk", "growth factor", "ap-1",
        "fos", "jun", "phosphatase", "sprouty"
    ],

    "myc_targets": [
        "myc", "ribosome", "translation", "nucleolus",
        "nucleotide", "biosynthesis", "rna processing",
        "cell growth"
    ],

    "p53_pathway": [
        "p53", "dna damage", "cell cycle arrest", "apoptosis",
        "stress response", "tp53", "checkpoint"
    ],

    "cell_cycle": [
        "cell cycle", "mitotic", "mitosis", "dna replication",
        "chromosome", "spindle", "proliferation", "division"
    ],

    "dna_damage_repair": [
        "dna repair", "double-strand break", "homologous recombination",
        "mismatch repair", "checkpoint", "fanconi", "damage response"
    ],

    "glycolysis_metabolic_rewiring": [
        "glycolysis", "glucose", "lactate", "pyruvate",
        "warburg", "hypoxia", "carbohydrate metabolism"
    ],

    "oxidative_phospho_mitochondrial": [
        "oxidative phosphorylation", "mitochondrial", "respiratory chain",
        "electron transport", "atp synthesis", "tca", "mitochondrion"
    ],

    "ecm_remodeling_invasion": [
        "extracellular matrix", "collagen", "matrix organization",
        "matrix remodeling", "integrin", "metalloproteinase",
        "invasion", "adhesion"
    ],

    "antigen_presentation_mhc_i": [
        "antigen processing", "mhc class i", "class i",
        "proteasome", "tap", "cytotoxic", "immune presentation"
    ],

    "antigen_presentation_mhc_ii": [
        "antigen processing", "mhc class ii", "class ii",
        "cd74", "ciita", "antigen presentation"
    ],

    "stemness_progenitor_like": [
        "stem", "progenitor", "development", "differentiation",
        "tumor initiating", "plasticity", "epithelial", "regeneration"
    ],

    "therapy_resist_stress_tolerance": [
        "drug", "xenobiotic", "transport", "efflux", "abc transporter",
        "apoptosis", "anti-apoptotic", "stress response",
        "oxidative stress", "heat shock", "detoxification"
    ],

    "tgfb_signaling": [
        "tgf", "smad", "transforming growth factor", "emt",
        "epithelial mesenchymal", "matrix", "fibrosis", "plasticity"
    ],

    "apoptosis": [
        "apoptosis", "programmed cell death", "caspase",
        "mitochondrial apoptosis", "intrinsic apoptotic",
        "extrinsic apoptotic"
    ],

    "senescence_sasp": [
        "senescence", "cell cycle arrest", "sasp",
        "inflammatory", "cytokine", "matrix remodeling",
        "stress-induced"
    ],

    "ap1_immediate_early_response": [
        "immediate early", "ap-1", "fos", "jun", "mapk",
        "stress response", "growth factor", "transcriptional response"
    ],
}


# Add keyword expectations for stricter subprograms by reusing parent logic.
EXPECTED_KEYWORDS_CANCER.update({
    "emt_tf_core": EXPECTED_KEYWORDS_CANCER["emt_invasion"],
    "emt_mesenchymal_core": EXPECTED_KEYWORDS_CANCER["emt_invasion"],
    "ecm_collagen_core": EXPECTED_KEYWORDS_CANCER["ecm_remodeling_invasion"],
    "ecm_protease_invasion_core": EXPECTED_KEYWORDS_CANCER["ecm_remodeling_invasion"],

    "nfkb_cytokine_core": EXPECTED_KEYWORDS_CANCER["inflammatory_signaling"],
    "chemokine_core": EXPECTED_KEYWORDS_CANCER["inflammatory_signaling"],

    "interferon_tf_core": EXPECTED_KEYWORDS_CANCER["interferon_response"],
    "interferon_isg_core": EXPECTED_KEYWORDS_CANCER["interferon_response"],

    "hif_hypoxia_core": EXPECTED_KEYWORDS_CANCER["hypoxia"],
    "hypoxia_glycolysis_component": EXPECTED_KEYWORDS_CANCER["glycolysis_metabolic_rewiring"],

    "nrf2_antioxidant_core": EXPECTED_KEYWORDS_CANCER["oxidative_stress"],
    "glutathione_redox_core": EXPECTED_KEYWORDS_CANCER["oxidative_stress"],

    "upr_sensor_tf_core": EXPECTED_KEYWORDS_CANCER["unfolded_protein_response"],
    "er_chaperone_folding_core": EXPECTED_KEYWORDS_CANCER["unfolded_protein_response"],

    "ras_mapk_feedback_core": EXPECTED_KEYWORDS_CANCER["KRAS_mapk_signaling"],
    "ap1_core": EXPECTED_KEYWORDS_CANCER["ap1_immediate_early_response"],

    "myc_biosynthesis_core": EXPECTED_KEYWORDS_CANCER["myc_targets"],
    "p53_arrest_apoptosis_core": EXPECTED_KEYWORDS_CANCER["p53_pathway"],

    "s_phase_replication_core": EXPECTED_KEYWORDS_CANCER["cell_cycle"],
    "g2m_mitotic_core": EXPECTED_KEYWORDS_CANCER["cell_cycle"],

    "homologous_recombination_core": EXPECTED_KEYWORDS_CANCER["dna_damage_repair"],
    "mismatch_checkpoint_repair_core": EXPECTED_KEYWORDS_CANCER["dna_damage_repair"],

    "glycolysis_core": EXPECTED_KEYWORDS_CANCER["glycolysis_metabolic_rewiring"],
    "lactate_pyruvate_core": EXPECTED_KEYWORDS_CANCER["glycolysis_metabolic_rewiring"],

    "mitochondrial_complex_i_core": EXPECTED_KEYWORDS_CANCER["oxidative_phospho_mitochondrial"],
    "mitochondrial_etc_atp_core": EXPECTED_KEYWORDS_CANCER["oxidative_phospho_mitochondrial"],

    "mhc_i_core": EXPECTED_KEYWORDS_CANCER["antigen_presentation_mhc_i"],
    "mhc_ii_core": EXPECTED_KEYWORDS_CANCER["antigen_presentation_mhc_ii"],

    "stemness_epithelial_core": EXPECTED_KEYWORDS_CANCER["stemness_progenitor_like"],
    "stemness_plasticity_component": EXPECTED_KEYWORDS_CANCER["stemness_progenitor_like"],

    "drug_efflux_core": EXPECTED_KEYWORDS_CANCER["therapy_resist_stress_tolerance"],
    "anti_apoptotic_resistance_core": EXPECTED_KEYWORDS_CANCER["therapy_resist_stress_tolerance"],
    "stress_tolerance_core": EXPECTED_KEYWORDS_CANCER["therapy_resist_stress_tolerance"],

    "tgfb_smad_core": EXPECTED_KEYWORDS_CANCER["tgfb_signaling"],
    "tgfb_emt_response_core": EXPECTED_KEYWORDS_CANCER["tgfb_signaling"],

    "intrinsic_apoptosis_core": EXPECTED_KEYWORDS_CANCER["apoptosis"],
    "executioner_caspase_core": EXPECTED_KEYWORDS_CANCER["apoptosis"],
    "anti_apoptotic_buffering_core": EXPECTED_KEYWORDS_CANCER["apoptosis"],

    "senescence_arrest_core": EXPECTED_KEYWORDS_CANCER["senescence_sasp"],
    "sasp_inflammatory_core": EXPECTED_KEYWORDS_CANCER["senescence_sasp"],

    "immediate_early_tf_core": EXPECTED_KEYWORDS_CANCER["ap1_immediate_early_response"],
    "mapk_feedback_immediate_core": EXPECTED_KEYWORDS_CANCER["ap1_immediate_early_response"],
})


In [ ]:
BASE_CANDIDATE_GENESET_LIBRARIES = [
    "GO_Biological_Process_2023",
    "GO_Cellular_Component_2023",
    "GO_Molecular_Function_2023",
    "Reactome_2022",
    "KEGG_2019_Mouse",
    "Mouse_Gene_Atlas",
    "PanglaoDB_Augmented_2021",
    "MGI_Mammalian_Phenotype_Level_4_2021",
]

CANCER_CANDIDATE_GENESET_LIBRARIES = [
    *BASE_CANDIDATE_GENESET_LIBRARIES,
    "MSigDB_Hallmark_2020",
]

In [ ]:
# Warning rules prevent overclaiming when programs are overlapping or mixed.
WORDING_WARNING_RULES_CANCER = [
    {
        "type": "overlap",
        "program_a": "emt_invasion",
        "program_b": "ecm_remodeling_invasion",
        "threshold": 0.25,
        "recommendation": (
            "These programs share matrix/invasion biology. Avoid treating EMT "
            "and ECM remodeling as fully independent if overlap is high."
        )
    },
    {
        "type": "overlap",
        "program_a": "hypoxia",
        "program_b": "glycolysis_metabolic_rewiring",
        "threshold": 0.20,
        "recommendation": (
            "Hypoxia and glycolysis may share metabolic genes. Interpret "
            "glycolysis as metabolic rewiring, not purely hypoxia-independent."
        )
    },
    {
        "type": "overlap",
        "program_a": "oxidative_stress",
        "program_b": "therapy_resist_stress_tolerance",
        "threshold": 0.20,
        "recommendation": (
            "Therapy resistance may partly reflect antioxidant/stress tolerance "
            "markers. Avoid overclaiming drug resistance without efflux or "
            "anti-apoptotic support."
        )
    },
    {
        "type": "overlap",
        "program_a": "KRAS_mapk_signaling",
        "program_b": "ap1_immediate_early_response",
        "threshold": 0.20,
        "recommendation": (
            "KRAS/MAPK and AP-1 immediate-early programs are mechanistically "
            "related. AP-1 may represent MAPK output or acute stimulation."
        )
    },
    {
        "type": "overlap",
        "program_a": "cell_cycle",
        "program_b": "dna_damage_repair",
        "threshold": 0.15,
        "recommendation": (
            "DNA repair can correlate with cycling because replication genes "
            "overlap with repair/checkpoint activity. Interpret repair separately "
            "from proliferation only if specific repair genes drive the signal."
        )
    },
    {
        "type": "overlap",
        "program_a": "senescence_sasp",
        "program_b": "inflammatory_signaling",
        "threshold": 0.15,
        "recommendation": (
            "SASP and inflammatory signaling share cytokine/chemokine genes. "
            "Elevated SASP scores should not automatically imply senescence "
            "unless arrest markers are also present."
        )
    },
    {
        "type": "overlap",
        "program_a": "p53_pathway",
        "program_b": "apoptosis",
        "threshold": 0.15,
        "recommendation": (
            "p53 and apoptosis programs overlap through pro-apoptotic genes. "
            "Interpret p53 as stress-response signaling, not necessarily cell death."
        )
    },
    {
        "type": "overlap",
        "program_a": "stemness_progenitor_like",
        "program_b": "emt_invasion",
        "threshold": 0.15,
        "recommendation": (
            "Stemness/progenitor-like programs containing Vim/Lgals1 may partly "
            "reflect plasticity or EMT-like features."
        )
    },
    {
        "type": "contains_any",
        "program": "therapy_resist_stress_tolerance",
        "genes": ["Abcb1a", "Abcb1b", "Abcc1", "Abcc3", "Abcg2"],
        "recommendation": (
            "Interpret as therapy-resistance-associated stress tolerance. "
            "Do not claim measured drug resistance without functional or "
            "treatment-response evidence."
        )
    },
    {
        "type": "contains_any",
        "program": "antigen_presentation_mhc_ii",
        "genes": ["Cd74", "Ciita", "H2-Aa", "H2-Ab1"],
        "recommendation": (
            "MHC-II genes can mark antigen-presenting immune cells as well as "
            "tumor-cell IFN-induced MHC-II states. Confirm malignant-cell origin "
            "before interpreting this as tumor-intrinsic MHC-II presentation."
        )
    },
    {
        "type": "contains_any",
        "program": "inflammatory_signaling",
        "genes": ["Il1b", "Ccl2", "Ccl5", "Cxcl1", "Cxcl2"],
        "recommendation": (
            "Inflammatory genes may be tumor-intrinsic or reflect immune/stromal "
            "contamination. Pair this score with tumor identity markers or "
            "copy-number/malignancy calls."
        )
    },
]

**Program validation routine**

In [19]:
OUTDIR_cp = os.path.join(commonpathlocal, commonpath,"cancer_program_biology_validation/")
os.makedirs(OUTDIR_cp, exist_ok=True)

results = run_program_validation(
    programs=PROGRAMS_MOUSE_CANCER_EXTENDED,
    expected_keywords=EXPECTED_KEYWORDS_CANCER,
    candidate_libraries=CANCER_CANDIDATE_GENESET_LIBRARIES,
    warning_rules=WORDING_WARNING_RULES_CANCER,
    outdir= OUTDIR_cp,
    organism="Mouse",
    pval_cutoff=0.05,
    title_prefix="Cancer program"
)


Libraries used:
  - GO_Biological_Process_2023
  - GO_Cellular_Component_2023
  - GO_Molecular_Function_2023
  - Reactome_2022
  - KEGG_2019_Mouse
  - Mouse_Gene_Atlas
  - PanglaoDB_Augmented_2021
  - MGI_Mammalian_Phenotype_Level_4_2021
  - MSigDB_Hallmark_2020
Running enrichment for: emt_invasion
Running enrichment for: epithelial_identity
Running enrichment for: inflammatory_signaling
Running enrichment for: interferon_response
Running enrichment for: hypoxia
Running enrichment for: oxidative_stress
Running enrichment for: unfolded_protein_response
Running enrichment for: KRAS_mapk_signaling
Running enrichment for: myc_targets
Running enrichment for: p53_pathway
Running enrichment for: cell_cycle
Running enrichment for: dna_damage_repair
Running enrichment for: glycolysis_metabolic_rewiring
Running enrichment for: oxidative_phospho_mitochondrial
Running enrichment for: ecm_remodeling_invasion
Running enrichment for: antigen_presentation_mhc_i
Running enrichment for: antigen_present

### Developmental programs

In [ ]:
# Original / curated programs
# These are your biological programs.

PROGRAMS_MOUSE = {
    "foregut_early_epithelial_identity": [
        "Sox2", "Sox9", "Foxa1", "Foxa2", "Hnf1b",
        "Krt8", "Krt18", "Epcam", "Cldn4"
    ],

    "ductal_differentiation": [
        "Krt19", "Krt7", "Sox9", "Cftr", "Muc1",
        "Mmp7", "Slc4a4", "Anxa4", "Cldn10", "Hnf1b"
    ],

    "acinar_differentiation": [
        "Prss1", "Prss2", "Cela2a", "Cela3b",
        "Cpa1", "Cpa2", "Cpb1", "Ctrb1", 
        "Reg1", "Reg2", "Pnlip", "Clps", "Amy2a5"
    ],

    "biliary_like_differentiation": [
        "Krt7", "Krt19", "Sox9", "Epcam", "Muc1",
        "Tff1", "Tff2", "Slc4a2", "Anxa4",
        "Cldn4", "Cldn10", "Hnf1b"
    ],

    "proliferation_cell_cycle": [
        "Mki67", "Top2a", "Pcna", "Mcm2", "Mcm3", "Mcm4",
        "Mcm5", "Mcm6", "Mcm7", "Cdk1", "Ccnb1",
        "Ccnb2", "Ube2c", "Birc5"
    ],

    "morphogenesis_epithelial_maturation": [
        "Epcam", "Krt8", "Krt18", "Krt19", "Cdh1",
        "Cldn3", "Cldn4", "Ocln", "Tjp1",
        "Itga6", "Lamc2"
    ],

    "secretory_digestive_enzyme_program": [
        "Prss1", "Prss2", "Cela2a", "Cela3b",
        "Cpa1", "Cpa2", "Cpb1", "Ctrb1", 
        "Pnlip", "Clps", "Amy2a5" 
    ],

    "progenitor_fetal_like_program": [
        "Sox9", "Prom1", "Aldh1a1", "Aldh1a3",
        "Nes", "Tff1", "Tff2",
        "S100a10", "Lgals1", "Vim"
    ],
    
    "neuroendocrine_like_peptide_hormone": [
        "Chga", "Chgb", "Scg2", "Scg3", "Scg5",
        "Snap25", "Syt1", "Syt4", "Syt13", "Rab3a", "Rab3b",
        "Neurod1", "Fev", "Isl1", "Pax6", "Nkx2-2",
        "Gip", "Cck", "Gast", "Tph1", "Penk", "Resp18"
    ],
}

In [ ]:
# Developmental epithelial programs: full + stricter subprograms

PROGRAMS_MOUSE_EXTENDED = {
    
    ########################################################

    **PROGRAMS_MOUSE,

    # Foregut / early epithelial stricter subprograms
    # Purpose:
    #   Test whether the full foregut score is supported by
    #   foregut/endoderm TFs, or only by generic epithelial genes.

    "foregut_endoderm_tf_core": [
        "Sox2", "Sox9", "Foxa1", "Foxa2", "Hnf1b"
    ],

    "foregut_fox_hnf_core": [
        "Foxa1", "Foxa2", "Hnf1b"
    ],

    "foregut_progenitor_component": [
        "Sox9", "Hnf1b"
    ],

    "early_epithelial_structural_core": [
        "Krt8", "Krt18", "Epcam", "Cldn4"
    ],


    ########################################################
    # Ductal / biliary stricter subprograms
    ########################################################
    # Purpose:
    #   Separate generic/shared ductal epithelium from the
    #   more biliary-like/mucinous component.

    "ductal_core_stricter": [
        "Krt19", "Krt7", "Cftr", "Muc1",
        "Mmp7", "Slc4a4", "Cldn10"
    ],

    "biliary_like_core_stricter": [
        "Tff1", "Tff2", "Slc4a2", "Anxa4", "Cldn4"
    ],

    "ductal_biliary_shared_epithelial": [
        "Krt7", "Krt19", "Sox9", "Epcam", "Muc1", "Hnf1b"
    ],


    ########################################################
    # Acinar / digestive-enzyme stricter subprograms
    ########################################################
    # Purpose:
    #   Test whether acinar_differentiation is broader than
    #   the enzyme-producing component.

    "acinar_enzyme_core": [
        "Prss1", "Prss2", "Cela2a", "Cela3b",
        "Cpa1", "Cpa2", "Cpb1", "Ctrb1",
        "Pnlip", "Clps", "Amy2a5"
    ],

    "acinar_regenerative_context": [
        "Reg1", "Reg2"
    ],


    ########################################################
    # Epithelial maturation stricter subprograms
    ########################################################
    # Purpose:
    #   Separate structural epithelial identity, junctional
    #   maturation, and basement-membrane/adhesion components.

    "epithelial_junction_core": [
        "Cdh1", "Cldn3", "Cldn4", "Ocln", "Tjp1"
    ],

    "epithelial_structural_core": [
        "Epcam", "Krt8", "Krt18", "Krt19"
    ],

    "basement_membrane_adhesion_core": [
        "Itga6", "Lamc2"
    ],


    ########################################################
    # Progenitor / fetal-like stricter subprograms
    ########################################################
    # Purpose:
    #   Test whether the full program is supported by a
    #   progenitor-like component, a fetal/mucinous component,
    #   or a mesenchymal/plasticity component.

    "progenitor_core": [
        "Sox9", "Prom1", "Aldh1a1", "Aldh1a3", "Nes"
    ],

    "fetal_mucinous_core": [
        "Tff1", "Tff2", "S100a10", "Lgals1"
    ],

    "mesenchymal_plasticity_component": [
        "Vim", "Lgals1"
    ],


    ########################################################
    # Neuroendocrine stricter subprograms
    ########################################################
    # Purpose:
    #   Test whether the full neuroendocrine-like program is
    #   supported by secretory machinery, TFs, and peptide
    #   hormone genes together.

    "neuroendocrine_secretory_core": [
        "Chga", "Chgb", "Scg2", "Scg3", "Scg5",
        "Snap25", "Syt1", "Syt4", "Syt13",
        "Rab3a", "Rab3b", "Resp18"
    ],

    "neuroendocrine_tf_core": [
        "Neurod1", "Fev", "Isl1", "Pax6", "Nkx2-2"
    ],

    "peptide_hormone_core": [
        "Gip", "Cck", "Gast", "Tph1", "Penk"
    ],
}

In [22]:
EXPECTED_KEYWORDS_DEVELOPMENT = {
    "foregut_early_epithelial_identity": [
        "foregut", "endoderm", "epithelial", "epithelium",
        "gut tube", "anterior foregut", "embryonic development",
        "organ development", "morphogenesis", "pancreas",
        "liver", "lung"
    ],

    "foregut_endoderm_tf_core": [
        "foregut", "endoderm", "gut tube", "anterior foregut",
        "embryonic development", "organ development",
        "pancreas", "liver", "lung"
    ],

    "foregut_fox_hnf_core": [
        "foregut", "endoderm", "gut tube",
        "pancreas", "liver", "lung", "organ development"
    ],

    "foregut_progenitor_component": [
        "progenitor", "epithelial", "endoderm",
        "pancreas", "duct", "organ development"
    ],

    "early_epithelial_structural_core": [
        "epithelial", "epithelium", "keratin",
        "cell junction", "tight junction", "cell adhesion"
    ],

    "ductal_differentiation": [
        "duct", "epithelial", "epithelium", "transport",
        "bicarbonate", "ion", "chloride", "pancreas"
    ],

    "ductal_core_stricter": [
        "duct", "epithelial", "epithelium", "transport",
        "bicarbonate", "ion", "chloride", "pancreas"
    ],

    "biliary_like_differentiation": [
        "bile", "biliary", "cholangiocyte", "duct",
        "epithelial", "epithelium", "mucin", "trefoil",
        "gastrointestinal", "liver"
    ],

    "biliary_like_core_stricter": [
        "bile", "biliary", "cholangiocyte", "mucin",
        "trefoil", "gastrointestinal", "epithelial"
    ],

    "ductal_biliary_shared_epithelial": [
        "epithelial", "epithelium", "duct", "pancreas"
    ],

    "acinar_differentiation": [
        "acinar", "exocrine", "pancreas", "pancreatic",
        "digestion", "digestive", "enzyme", "protease",
        "amylase", "lipase", "secretion"
    ],

    "secretory_digestive_enzyme_program": [
        "digestion", "digestive", "enzyme", "protease",
        "amylase", "lipase", "secretion", "exocrine",
        "pancreas", "pancreatic"
    ],

    "acinar_enzyme_core": [
        "digestion", "digestive", "enzyme", "protease",
        "amylase", "lipase", "exocrine", "pancreas"
    ],

    "acinar_regenerative_context": [
        "regeneration", "injury", "stress response",
        "inflammatory response", "pancreas"
    ],

    "proliferation_cell_cycle": [
        "cell cycle", "mitotic", "mitosis", "dna replication",
        "chromosome", "spindle", "proliferation", "division"
    ],

    "morphogenesis_epithelial_maturation": [
        "epithelial", "epithelium", "cell junction",
        "tight junction", "adherens", "polarity", "adhesion",
        "morphogenesis", "basement membrane"
    ],

    "epithelial_junction_core": [
        "cell junction", "tight junction", "adherens",
        "epithelial", "epithelium", "polarity", "adhesion"
    ],

    "epithelial_structural_core": [
        "epithelial", "epithelium", "keratin",
        "intermediate filament", "cell adhesion"
    ],

    "basement_membrane_adhesion_core": [
        "basement membrane", "extracellular matrix",
        "adhesion", "integrin", "laminin"
    ],

    "progenitor_fetal_like_program": [
        "progenitor", "stem", "development", "fetal",
        "embryonic", "differentiation", "epithelial",
        "regeneration", "morphogenesis"
    ],

    "progenitor_core": [
        "progenitor", "stem", "development",
        "differentiation", "embryonic"
    ],

    "fetal_mucinous_core": [
        "mucin", "trefoil", "gastrointestinal",
        "epithelial", "fetal", "development"
    ],

    "mesenchymal_plasticity_component": [
        "mesenchymal", "migration", "extracellular matrix",
        "adhesion", "wound", "fibroblast"
    ],

    "neuroendocrine_like_peptide_hormone": [
        "neuroendocrine", "endocrine", "hormone",
        "peptide", "secretory granule", "synaptic vesicle",
        "regulated secretion", "neuron", "neuropeptide"
    ],

    "neuroendocrine_secretory_core": [
        "secretory granule", "synaptic vesicle",
        "regulated secretion", "neuroendocrine",
        "exocytosis", "vesicle"
    ],

    "neuroendocrine_tf_core": [
        "endocrine", "neuroendocrine", "neuron",
        "differentiation", "development", "transcription"
    ],

    "peptide_hormone_core": [
        "hormone", "peptide", "neuropeptide",
        "endocrine", "gastrointestinal"
    ],
}

In [25]:
DEVELOPMENT_CANDIDATE_GENESET_LIBRARIES = BASE_CANDIDATE_GENESET_LIBRARIES

In [ ]:
# Developmental epithelial wording-warning rules
# ============================================================

DEVELOPMENT_WARNING_RULES = [
    ########################################################
    # Foregut / epithelial identity
    ########################################################

    {
        "type": "overlap",
        "program_a": "foregut_early_epithelial_identity",
        "program_b": "early_epithelial_structural_core",
        "threshold": 0.35,
        "recommendation": (
            "The foregut early epithelial program contains a substantial generic "
            "epithelial component. Do not interpret high scores as definitive "
            "foregut/endoderm identity unless the foregut/endoderm TF component "
            "is also supported."
        )
    },

    {
        "type": "contains_any",
        "program": "foregut_early_epithelial_identity",
        "genes": ["Krt8", "Krt18", "Epcam", "Cldn4"],
        "recommendation": (
            "These genes support epithelial identity but are not foregut-specific. "
            "Use wording such as 'early epithelial state with foregut/endoderm-"
            "associated transcriptional features' rather than 'foregut identity' "
            "unless TF-core terms are enriched."
        )
    },

    ########################################################
    # Ductal / biliary-like relationship
    ########################################################

    {
        "type": "overlap",
        "program_a": "ductal_differentiation",
        "program_b": "biliary_like_differentiation",
        "threshold": 0.30,
        "recommendation": (
            "Ductal and biliary-like programs share a ductal epithelial backbone. "
            "Interpret biliary-like scores as ductal/biliary-like or mucinous "
            "ductal epithelial features, not bona fide biliary lineage conversion."
        )
    },

    {
        "type": "overlap",
        "program_a": "biliary_like_differentiation",
        "program_b": "ductal_biliary_shared_epithelial",
        "threshold": 0.40,
        "recommendation": (
            "The biliary-like program may be driven by shared ductal epithelial "
            "markers. Check whether the stricter biliary-like core contributes "
            "supporting enrichment before making a biliary-like claim."
        )
    },

    {
        "type": "contains_any",
        "program": "biliary_like_differentiation",
        "genes": ["Tff1", "Tff2", "Slc4a2", "Anxa4"],
        "recommendation": (
            "The biliary-like label is supported only if mucinous/gastro-biliary "
            "features contribute to enrichment. Otherwise, use the safer term "
            "'ductal/mucinous epithelial program'."
        )
    },

    ########################################################
    # Acinar / digestive enzyme redundancy
    ########################################################

    {
        "type": "overlap",
        "program_a": "acinar_differentiation",
        "program_b": "secretory_digestive_enzyme_program",
        "threshold": 0.65,
        "recommendation": (
            "The acinar differentiation and secretory digestive enzyme programs "
            "are highly overlapping. Treat them as closely related acinar/exocrine "
            "enzyme programs rather than independent biological states."
        )
    },

    {
        "type": "overlap",
        "program_a": "acinar_differentiation",
        "program_b": "acinar_enzyme_core",
        "threshold": 0.65,
        "recommendation": (
            "The acinar differentiation program is mostly an enzyme-production "
            "program. Only describe it as broader acinar maturation if non-enzyme "
            "or contextual markers add interpretable support."
        )
    },

    ########################################################
    # Proliferation
    ########################################################

    {
        "type": "contains_any",
        "program": "proliferation_cell_cycle",
        "genes": ["Mki67", "Top2a", "Pcna", "Mcm2", "Cdk1", "Ccnb1", "Ube2c"],
        "recommendation": (
            "This is a cell-state program, not a lineage program. Interpret it "
            "orthogonally to ductal, acinar, epithelial, or neuroendocrine identity."
        )
    },

    ########################################################
    # Epithelial morphogenesis / maturation
    ########################################################

    {
        "type": "overlap",
        "program_a": "morphogenesis_epithelial_maturation",
        "program_b": "epithelial_structural_core",
        "threshold": 0.30,
        "recommendation": (
            "The epithelial maturation program contains a structural epithelial "
            "backbone. Avoid treating it as lineage-specific differentiation; it "
            "mainly reflects epithelial organization and maturation."
        )
    },

    {
        "type": "overlap",
        "program_a": "morphogenesis_epithelial_maturation",
        "program_b": "epithelial_junction_core",
        "threshold": 0.30,
        "recommendation": (
            "The program contains a strong junctional component. Interpret high "
            "scores as epithelial organization/junctional maturation, not a "
            "specific developmental lineage."
        )
    },

    ########################################################
    # Progenitor / fetal-like mixed composition
    ########################################################

    {
        "type": "contains_any",
        "program": "progenitor_fetal_like_program",
        "genes": ["Vim", "Lgals1", "Tff1", "Tff2", "S100a10"],
        "recommendation": (
            "This program mixes progenitor-associated, fetal/mucinous, and "
            "plasticity-associated markers. Do not describe it as a pure progenitor "
            "signature. Use 'progenitor/fetal-like or plasticity-associated'."
        )
    },

    {
        "type": "overlap",
        "program_a": "progenitor_fetal_like_program",
        "program_b": "mesenchymal_plasticity_component",
        "threshold": 0.15,
        "recommendation": (
            "The progenitor/fetal-like program contains mesenchymal/plasticity "
            "features. Interpret elevated scores alongside epithelial and "
            "mesenchymal/stromal markers."
        )
    },

    {
        "type": "overlap",
        "program_a": "progenitor_fetal_like_program",
        "program_b": "fetal_mucinous_core",
        "threshold": 0.20,
        "recommendation": (
            "The progenitor/fetal-like program may be driven by fetal/mucinous "
            "features rather than progenitor identity alone."
        )
    },

    ########################################################
    # Neuroendocrine-like
    ########################################################

    {
        "type": "overlap",
        "program_a": "neuroendocrine_like_peptide_hormone",
        "program_b": "neuroendocrine_secretory_core",
        "threshold": 0.40,
        "recommendation": (
            "The neuroendocrine-like program may be driven strongly by secretory "
            "machinery. Interpret as neuroendocrine-like most confidently when "
            "secretory machinery, endocrine TFs, and peptide hormone components "
            "are all supported."
        )
    },

    {
        "type": "contains_any",
        "program": "neuroendocrine_like_peptide_hormone",
        "genes": ["Gip", "Cck", "Gast", "Tph1", "Penk"],
        "recommendation": (
            "Peptide hormone genes support an enteroendocrine/neuroendocrine-like "
            "interpretation, but sparse hormone expression should be checked "
            "against secretory machinery and endocrine TF support."
        )
    },
]

**Program validation routine**

In [24]:
# ============================================================
# Developmental epithelial validation run
# ============================================================

OUTDIR_p = os.path.join(commonpathlocal, commonpath,"development_program_biology_validation/")
os.makedirs(OUTDIR_p, exist_ok=True)

results = run_program_validation(
    programs=PROGRAMS_MOUSE_EXTENDED,
    expected_keywords=EXPECTED_KEYWORDS_DEVELOPMENT,
    candidate_libraries=DEVELOPMENT_CANDIDATE_GENESET_LIBRARIES,
    warning_rules=DEVELOPMENT_WARNING_RULES,
    outdir=OUTDIR_p,
    organism="Mouse",
    pval_cutoff=0.05,
    title_prefix="Developmental epithelial program"
)


Libraries used:
  - GO_Biological_Process_2023
  - GO_Cellular_Component_2023
  - GO_Molecular_Function_2023
  - Reactome_2022
  - KEGG_2019_Mouse
  - Mouse_Gene_Atlas
  - PanglaoDB_Augmented_2021
  - MGI_Mammalian_Phenotype_Level_4_2021
Running enrichment for: foregut_early_epithelial_identity
Running enrichment for: ductal_differentiation
Running enrichment for: acinar_differentiation
Running enrichment for: biliary_like_differentiation
Running enrichment for: proliferation_cell_cycle
Running enrichment for: morphogenesis_epithelial_maturation
Running enrichment for: secretory_digestive_enzyme_program
Running enrichment for: progenitor_fetal_like_program
Running enrichment for: neuroendocrine_like_peptide_hormone
Running enrichment for: foregut_endoderm_tf_core
Running enrichment for: foregut_fox_hnf_core
Running enrichment for: foregut_progenitor_component
Running enrichment for: early_epithelial_structural_core
Running enrichment for: ductal_core_stricter
Running enrichment for: b